In [1]:
import json
import random
import copy
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT  = Path.home() / "icidea_llm_ids"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

LABEL_MAP = {0: "NORMAL", 1: "DOS", 2: "FUZZY", 3: "GEAR", 4: "RPM"}
SEEDS = [42, 123, 2024]
SAMPLES_PER_CLASS = 20  # 20 per class = 100 per seed

# Load data
classifier_results = pd.read_parquet(
    ARTIFACTS_DIR / "section9a_classifier_results.parquet"
)
perturbed_dataset = pd.read_parquet(
    ARTIFACTS_DIR / "section10_perturbed_dataset.parquet"
)
windows_df = pd.read_parquet(
    ARTIFACTS_DIR / "section7_windows_with_text.parquet"
)

print(f"✓ Data loaded")
print(f"  Classifier results: {len(classifier_results)}")
print(f"  Perturbed dataset: {len(perturbed_dataset)}")

# Build lookup
idx_to_ts   = classifier_results["window_start_ts"].to_dict()
idx_to_text = classifier_results["text"].to_dict()

def frames_to_text(frames):
    n = len(frames)
    lines = [f"CAN Bus Telemetry Sequence ({n} frames):"]
    for i, frame in enumerate(frames, start=1):
        can_id_hex = f"{frame['can_id']:04X}"
        dlc = frame["dlc"]
        data_bytes = frame["data"][:dlc]
        data_hex = " ".join(f"{b:02X}" for b in data_bytes)
        ts = frame["timestamp"]
        lines.append(
            f"[{i:03d}] T={ts:.3f} ID={can_id_hex} "
            f"DLC={dlc} DATA={data_hex}"
        )
    return "\n".join(lines)

def perturb_p1(frames, seed):
    rng = random.Random(seed)
    perturbed = copy.deepcopy(frames)
    target_idx = len(frames) // 2
    frame = perturbed[target_idx]
    byte_idx = rng.randint(0, 7)
    original_val = frame["data"][byte_idx]
    delta = rng.choice([-1, 1])
    new_val = (original_val + delta) % 256
    frame["data"][byte_idx] = new_val
    return perturbed, byte_idx, original_val, new_val

# Build seed datasets
all_seed_records = []

for seed in SEEDS:
    np.random.seed(seed)
    print(f"\nBuilding dataset for seed {seed}...")

    seed_records = []
    for lbl, name in LABEL_MAP.items():
        # Sample windows for this class
        class_rows = classifier_results[
            classifier_results["label"] == lbl
        ].sample(n=SAMPLES_PER_CLASS, random_state=seed)

        for _, row in class_rows.iterrows():
            window_idx = row.name
            text = row["text"]
            ts = row["window_start_ts"]

            # Get frames
            match = windows_df[windows_df["window_start_ts"] == ts]
            if len(match) == 0:
                continue
            frames = json.loads(match.iloc[0]["frames_json"])

            # Apply P1 perturbation with this seed
            p1_frames, byte_idx, orig, mutated = perturb_p1(frames, seed)
            p1_text = frames_to_text(p1_frames)

            seed_records.append({
                "seed":          seed,
                "window_idx":    window_idx,
                "true_label":    lbl,
                "true_label_name": name,
                "original_text": text,
                "p1_text":       p1_text,
                "perturbation":  f"frame7_byte{byte_idx}_{orig}to{mutated}",
            })

    print(f"  Records built: {len(seed_records)}")
    all_seed_records.extend(seed_records)

seeds_df = pd.DataFrame(all_seed_records)
print(f"\n✓ Total records: {len(seeds_df)}")
print(f"  Per seed: {len(seeds_df) // len(SEEDS)}")
print(f"\nSeed distribution:")
print(seeds_df["seed"].value_counts().to_string())

# Save for Kaggle upload
save_path = ARTIFACTS_DIR / "task2_seed_datasets.parquet"
seeds_df.to_parquet(save_path, index=False)
size_mb = save_path.stat().st_size / 1e6
print(f"\n✓ Saved to {save_path}")
print(f"  Size: {size_mb:.2f} MB")

✓ Data loaded
  Classifier results: 1500
  Perturbed dataset: 2500

Building dataset for seed 42...
  Records built: 100

Building dataset for seed 123...
  Records built: 100

Building dataset for seed 2024...
  Records built: 100

✓ Total records: 300
  Per seed: 100

Seed distribution:
seed
42      100
123     100
2024    100

✓ Saved to /Users/deepakpatnaik/icidea_llm_ids/artifacts/task2_seed_datasets.parquet
  Size: 0.15 MB
